# 05 · MCP integration (prototype)
**Objective:** exercise the official MCP Python SDK with a real model choosing a tool call: tool discovery, typed
input, a permission check inside the tool, and tenant scoping. Phase 7 builds the production server on real services.

The tool reads the fictional dataset; it is an in-memory demo server, not a deployed one.

In [ ]:
# Setup: works locally and on Google Colab. Never put secrets in notebook cells.
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = ""  # Set to your clone URL. On Colab, add HF_TOKEN under Secrets to use hosted models.
    if not Path("salesiq").exists():
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL in this cell to clone the SalesIQ repository.")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "salesiq"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml", "pydantic", "mcp"], check=True)
    ROOT = Path("salesiq").resolve()
    from google.colab import userSecrets
    try:
        os.environ["HF_TOKEN"] = userSecrets.get("HF_TOKEN")
    except userSecrets.SecretNotFoundError:
        print("No HF_TOKEN secret; only a local model server can be used.")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())
sys.path.insert(0, str(ROOT / "apps" / "api"))
os.environ["SALESIQ_RESEARCH_DIR"] = str(ROOT / "research")

from app.agents.llm import LLMError, ModelRegistry

registry = ModelRegistry()
status = {s.key: s for s in registry.status()}
CANDIDATES = ["qwen3-4b", "qwen2.5-7b", "mistral-7b", "hf-qwen3-8b"]
MODEL = next((k for k in CANDIDATES if status[k].available), None)
if MODEL is None:
    # There is deliberately no mock: results must come from a real model.
    raise RuntimeError("No model is reachable. Start Ollama and pull a model, or set HF_TOKEN.\n"
                       + "\n".join(f"{s.key}: {s.reason}" for s in status.values() if not s.available))
print(f"Using real model: {MODEL} ({registry.model(MODEL).model})")

In [ ]:
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session
from pydantic import BaseModel
from app.agents.evaluation.runner import Dataset

leads = Dataset.load("train").prospects
# Tenant and role are set by trusted server code, never by the model or the tool arguments.
SESSION = {"org": "org-a", "role": "analyst"}
TENANT_OF = {r["id"]: ("org-a" if i % 2 == 0 else "org-b") for i, r in enumerate(leads)}

server = FastMCP("salesiq-demo")

@server.tool()
def search_leads(query: str, limit: int = 3) -> list[dict]:
    "Read-only search over leads the caller may see."
    q = query.lower()
    hits = [r for r in leads if TENANT_OF[r["id"]] == SESSION["org"] and (q in r["note"].lower() or q in r["structured"]["industry"].lower())]
    return [{"id": r["id"], "company": r["structured"]["company"], "industry": r["structured"]["industry"]} for r in hits[:min(limit, 10)]]

@server.tool()
def delete_lead(lead_id: str) -> str:
    "Destructive. Requires the delete permission, checked here rather than by the model."
    if SESSION["role"] not in ("owner", "admin", "manager"):
        raise PermissionError("role may not delete leads")
    return "deleted"

In [ ]:
class ToolChoice(BaseModel):
    tool: str
    query: str

async with create_connected_server_and_client_session(server._mcp_server) as client:
    tools = (await client.list_tools()).tools
    print("Discovered:", [t.name for t in tools])
    catalog = [{"name": t.name, "description": t.description} for t in tools]
    prompt = f"Tools: {json.dumps(catalog)}\nRequest: find fintech leads. Choose the tool and a one-word query."
    choice, res = registry.structured(MODEL, [{"role": "user", "content": prompt}], ToolChoice, max_tokens=60)
    print("Model chose:", choice, f"({res.latency_ms:.0f} ms)")
    allowed = {t.name for t in tools}
    assert choice.tool in allowed, "Tool names are validated against the catalog"
    result = await client.call_tool("search_leads", {"query": choice.query})
    print("Result:", result.content[0].text if result.content else "no matches")

    # Authorization is enforced inside the tool, whatever the model asks for.
    denied = await client.call_tool("delete_lead", {"lead_id": "train-001"})
    print("Delete as analyst -> isError:", denied.isError, "|", denied.content[0].text)
    assert denied.isError

**Expected:** two discovered tools, a model-chosen search, and a refused delete because the session role is `analyst`.

**Limitations:** no transport authentication, audit log, or real services yet; those come with Phase 7. Model output is only
ever treated as a *request* that server code validates.